# 예제 franka_ex09: FR3 충돌 객체 (Collision Objects)

`PlanningScene` 에 박스 / 원기둥 장애물을 넣고, FR3 가 그 사이를 빠져나가는 경로를 찾도록 한다.

## 이 노트북이 강조하는 것 — `PlanningScene` 에 충돌 객체 동적 관리

ex03~08 의 모든 plan 은 **robot 만 있는 빈 world** 에서 일어났다. ex09 는 처음으로
**world 에 장애물을 추가**해서 플래너가 그걸 피해 가도록 한다.

| 단계 | 도구 |
|---|---|
| 객체 만들기 | `CollisionObject` + `SolidPrimitive(BOX/CYLINDER)` |
| Scene 변경 | `apply_planning_scene` 서비스 + `PlanningScene(is_diff=True)` |
| 추가 / 제거 | `CollisionObject.operation` 에 `ADD` / `REMOVE` |
| 일괄 제거 | `id=''` + `REMOVE` (관용 패턴) |

**플래닝 자체는 변경 없음** — `plan_to_pose_goal()` 만 부르면 플래너가 자동으로 Scene 의
충돌 객체를 피한다. ex09 의 본질은 *plan 하기 전에 world 에 무엇이 있는지 알려 주는 단계가
추가됐다*는 것.

`is_diff` 플래그가 핵심 — `True` 면 기존 Scene 위에 누적 / 제거, `False` 면 통째 교체.
이 예제는 항상 `is_diff=True` 로 보내 단계마다 객체 한 개를 더하고 빼면서 진행.

## 이전 예제와의 관계

ex07 의 `plan_viz_execute()` 패턴 (plan-only → FK 미리보기 → execute) 을 그대로 재사용.
EE 경로가 RViz 에 그려지므로 **장애물을 어떻게 우회하는지**가 한눈에 보인다 —
원본 노트북은 마커 publisher 만 만들고 쓰지 않았지만, 본 재구성에서는 의미 있게 활용.

## 노트북 구성
1. **로봇 상수**
2. **핵심 — `PlanningScene` 에 충돌 객체 추가/제거** ← 이 노트북의 본질
3. **핵심을 쓰기 위한 설정** — ROS init, 클라이언트 (apply_planning_scene 포함), SRDF, Pose / MoveGroup / FK / 마커
4. **시나리오** — 낮은 벽 / 옆 기둥 / 정면 원기둥

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 에서:
- **Planning Scene Display** 가 켜져 있어야 충돌 객체 (박스 / 원기둥) 가 보인다
- **`MarkerArray` Display** 를 추가하고 Topic 을 `/collision_demo_markers` 로 설정 — EE 경로가 보인다
- Fixed Frame 은 `fr3_link0`

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab franka_ex09_collision_objects.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 로봇 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/collision_demo_markers'

## 2. 핵심 — `PlanningScene` 에 충돌 객체 동적 관리

이 노트북에서 가장 먼저 정의해야 하는 함수들.

- `make_box()` / `make_cylinder()` — `CollisionObject` 빌더
- `apply_diff()` — `PlanningScene(is_diff=True)` 로 변경 적용
- `add_object()` / `remove_object()` / `clear_all()` — 편의 wrapper

함수들은 setup 셀에서 만드는 `node` / `scene_client` 를 globals 로 참조한다.
함수 *정의* 시점엔 lookup 하지 않으므로 객체가 아직 없어도 OK — 호출은 4 절(시나리오) 에서.

### 2-1. 핵심에 필요한 import

In [ ]:
import rclpy
from moveit_msgs.srv import ApplyPlanningScene
from moveit_msgs.msg import PlanningScene, CollisionObject
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Pose, Point, Quaternion

### 2-2. `make_box()` / `make_cylinder()` — `CollisionObject` 빌더

`CollisionObject` 한 개에 `SolidPrimitive` 도형 + `primitive_poses` 위치를 채운다.
도형 종류:
- `BOX`: `dimensions = [x, y, z]` (m)
- `CYLINDER`: `dimensions = [height, radius]` (m) — 순서 주의

`operation = ADD` 가 핵심 — 이 객체가 Scene 에 들어간다는 의미.

In [ ]:
def make_box(object_id: str, position, dimensions,
             frame_id: str = None) -> CollisionObject:
    co = CollisionObject()
    co.header.frame_id = frame_id or REFERENCE_FRAME
    co.id = object_id
    co.operation = CollisionObject.ADD
    box = SolidPrimitive()
    box.type = SolidPrimitive.BOX
    box.dimensions = list(dimensions)
    co.primitives.append(box)
    pose = Pose()
    pose.position = Point(x=position[0], y=position[1], z=position[2])
    pose.orientation = Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)
    co.primitive_poses.append(pose)
    return co


def make_cylinder(object_id: str, position, height: float, radius: float,
                  frame_id: str = None) -> CollisionObject:
    co = CollisionObject()
    co.header.frame_id = frame_id or REFERENCE_FRAME
    co.id = object_id
    co.operation = CollisionObject.ADD
    cyl = SolidPrimitive()
    cyl.type = SolidPrimitive.CYLINDER
    cyl.dimensions = [height, radius]
    co.primitives.append(cyl)
    pose = Pose()
    pose.position = Point(x=position[0], y=position[1], z=position[2])
    pose.orientation = Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)
    co.primitive_poses.append(pose)
    return co

### 2-3. `apply_diff()` + 추가/제거 wrapper

`PlanningScene(is_diff=True)` 로 보내면 기존 Scene 에 **누적 / 제거** 된다.
`is_diff=False` 면 Scene 을 통째 교체 (보통 안 씀 — 다른 객체까지 다 지워짐).

- `add_object(co)`: ADD operation 객체 한 개 보냄
- `remove_object(id)`: REMOVE operation 객체 한 개 보냄
- `clear_all()`: `id=''` + REMOVE — 관용적으로 모든 객체 일괄 제거

In [ ]:
def apply_diff(world_objects=None) -> bool:
    '''PlanningScene(is_diff=True) 로 world.collision_objects 변경.'''
    scene = PlanningScene()
    scene.is_diff = True
    if world_objects:
        scene.world.collision_objects = list(world_objects)
    req = ApplyPlanningScene.Request()
    req.scene = scene
    fut = scene_client.call_async(req)
    rclpy.spin_until_future_complete(node, fut)
    return fut.result().success


def add_object(co: CollisionObject) -> bool:
    return apply_diff([co])


def remove_object(object_id: str, frame_id: str = None) -> bool:
    co = CollisionObject()
    co.header.frame_id = frame_id or REFERENCE_FRAME
    co.id = object_id
    co.operation = CollisionObject.REMOVE
    return apply_diff([co])


def clear_all() -> bool:
    '''id="" + REMOVE 는 관용적으로 모든 객체 일괄 제거.'''
    co = CollisionObject()
    co.header.frame_id = REFERENCE_FRAME
    co.id = ''
    co.operation = CollisionObject.REMOVE
    return apply_diff([co])

## 3. 핵심을 쓰기 위한 설정

위 핵심 함수들이 참조하는 객체와 보조 헬퍼.

- ROS 2 초기화 / 노드 / 액션·서비스 클라이언트 (`apply_planning_scene` 포함) / `joint_states` 구독 / 마커 퍼블리셔 / FK 클라이언트
- 서버·서비스·`/joint_states` 준비 대기
- SRDF `ready` 자세
- Pose 헬퍼
- MoveGroup 빌딩블록
- ex07 패턴 — `plan_to_*_goal()` / `trajectory_to_ee_path()` / `publish_ee_path()` / `execute_trajectory()` / `plan_viz_execute()`

### 3-1. ROS 2 초기화 + 노드 + 클라이언트

In [ ]:
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.srv import GetPositionFK
from visualization_msgs.msg import MarkerArray

try:
    rclpy.init()
except RuntimeError:
    pass

node = Node(
    'franka_ex09_collision_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client    = ActionClient(node, MoveGroup, 'move_action')
execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
fk_client      = node.create_client(GetPositionFK, 'compute_fk')
scene_client   = node.create_client(ApplyPlanningScene, 'apply_planning_scene')
marker_pub     = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex09 노트북 노드 생성 완료 ===')

### 3-2. 액션 / 서비스 / `/joint_states` 준비 대기

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not execute_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')
    if not fk_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('compute_fk 서비스 연결 실패')
    if not scene_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('apply_planning_scene 서비스 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info(
        'move/execute action + compute_fk + apply_planning_scene + /joint_states 준비됨'
    )

wait_for_ready()

### 3-3. SRDF 에서 `ready` 자세 읽어오기

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

### 3-4. Pose 헬퍼 — Euler ↔ Quaternion

In [ ]:
import math
import tf_transformations

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

### 3-5. MoveGroup 빌딩블록 — Constraints / `MotionPlanRequest`

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes, RobotState,
)
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

### 3-6. ex07 패턴 — `plan_to_*_goal()` + FK + `execute_trajectory()`

**플래너는 Scene 의 충돌 객체를 자동으로 피한다.** 본 예제에서 이전과 다른 부분은 2 절의
Scene 변경뿐 — 플래닝 / 실행 워크플로는 ex07 그대로.

In [ ]:
def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only,
        replan=not plan_only,
        replan_attempts=3 if not plan_only else 0,
    )
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory


def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj


def plan_to_pose_goal(pose, vel: float = 0.3, acc: float = 0.3,
                      plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj


def trajectory_to_ee_path(trajectory, max_points: int = 60):
    jt = trajectory.joint_trajectory
    total = len(jt.points)
    if total == 0:
        return []
    step = max(1, total // max_points)
    indices = list(range(0, total, step))
    if indices[-1] != total - 1:
        indices.append(total - 1)
    pts = []
    for idx in indices:
        req = GetPositionFK.Request()
        req.header.frame_id = REFERENCE_FRAME
        req.fk_link_names = [END_EFFECTOR_LINK]
        rs = RobotState()
        rs.joint_state.name = list(jt.joint_names)
        rs.joint_state.position = list(jt.points[idx].positions)
        req.robot_state = rs
        fut = fk_client.call_async(req)
        rclpy.spin_until_future_complete(node, fut)
        resp = fut.result()
        if resp and resp.error_code.val == MoveItErrorCodes.SUCCESS and resp.pose_stamped:
            p = resp.pose_stamped[0].pose.position
            pts.append((p.x, p.y, p.z))
    return pts


def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

### 3-7. EE 경로 + 시작/목표 마커 + `plan_viz_execute()`

| 마커 | 의미 |
|---|---|
| 주황 LINE_STRIP (`COLOR_EE_PATH`) | 실행 직전 EE 경로 |
| 노란 SPHERE (`COLOR_START_PT`) | 시작점 |
| 색깔 SPHERE (시나리오별) | 목표점 |
| 흰 텍스트 | 라벨 (`Start` / 시나리오 라벨) |

`publish_endpoints()` 한 번 호출 = 새 시작/목표 두 sphere + 라벨 묶음 발행.
ns 가 고정 (`wp`, `wp_text`) 이라 시나리오 바뀔 때 자동으로 덮어써진다.
EE 경로 (`ee_path` ns) 와는 별개라 같이 보인다.

In [ ]:
from std_msgs.msg import ColorRGBA
from visualization_msgs.msg import Marker

COLOR_EE_PATH      = ColorRGBA(r=1.0, g=0.5, b=0.0, a=0.95)   # 주황
COLOR_START_PT     = ColorRGBA(r=1.0, g=0.9, b=0.0, a=0.95)   # 노랑
COLOR_GOAL_RED     = ColorRGBA(r=0.9, g=0.2, b=0.2, a=0.95)   # 빨강 (벽)
COLOR_GOAL_BLUE    = ColorRGBA(r=0.2, g=0.5, b=0.9, a=0.95)   # 파랑 (기둥)
COLOR_GOAL_PURPLE  = ColorRGBA(r=0.7, g=0.2, b=0.7, a=0.95)   # 보라 (원기둥)
COLOR_TEXT         = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)
_markers = MarkerArray()


def publish_ee_path(ee_points, color=None):
    if not ee_points:
        return
    if color is None:
        color = COLOR_EE_PATH
    stamp = node.get_clock().now().to_msg()
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = 'ee_path'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = 0.008
    line.color = color
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in ee_points]
    _markers.markers = [m for m in _markers.markers
                        if (m.ns, m.id) != ('ee_path', 0)]
    _markers.markers.append(line)
    marker_pub.publish(_markers)


def publish_endpoints(start_pos, goal_pos, goal_label: str, goal_color):
    '''시작점(노랑) + 목표(color) sphere + 라벨. 이전 endpoint 마커는 덮어씀.'''
    stamp = node.get_clock().now().to_msg()
    _markers.markers = [m for m in _markers.markers
                        if m.ns not in {'wp', 'wp_text'}]
    for i, (pos, color, label) in enumerate([
        (start_pos, COLOR_START_PT, 'Start'),
        (goal_pos,  goal_color,     goal_label),
    ]):
        s = Marker()
        s.header.frame_id = REFERENCE_FRAME
        s.header.stamp = stamp
        s.ns = 'wp'
        s.id = i
        s.type = Marker.SPHERE
        s.action = Marker.ADD
        s.pose.position = Point(x=pos[0], y=pos[1], z=pos[2])
        s.pose.orientation.w = 1.0
        s.scale = Vector3(x=0.045, y=0.045, z=0.045)
        s.color = color
        t = Marker()
        t.header.frame_id = REFERENCE_FRAME
        t.header.stamp = stamp
        t.ns = 'wp_text'
        t.id = i
        t.type = Marker.TEXT_VIEW_FACING
        t.action = Marker.ADD
        t.pose.position = Point(x=pos[0], y=pos[1], z=pos[2] + 0.08)
        t.pose.orientation.w = 1.0
        t.scale.z = 0.05
        t.color = COLOR_TEXT
        t.text = label
        _markers.markers.extend([s, t])
    marker_pub.publish(_markers)


def plan_viz_execute(pose, vel: float = 0.3, label: str = '') -> bool:
    ok, traj = plan_to_pose_goal(pose, vel=vel, acc=vel, plan_time=10.0)
    if not ok or traj is None:
        node.get_logger().error(f'{label}: 계획 실패 (장애물 충돌? IK 실패?)')
        return False
    pts = trajectory_to_ee_path(traj)
    if pts:
        publish_ee_path(pts)
    return execute_trajectory(traj)


def plan_viz_execute_joint(joint_values: dict, vel: float = 0.3,
                           label: str = '') -> bool:
    ok, traj = plan_to_joint_goal(joint_values, vel=vel, acc=vel)
    if not ok or traj is None:
        node.get_logger().error(f'{label}: 계획 실패')
        return False
    pts = trajectory_to_ee_path(traj)
    if pts:
        publish_ee_path(pts)
    return execute_trajectory(traj)

## 4. 시나리오 — 장애물 추가 → 회피 → 제거

각 시나리오의 패턴은 동일:

1. `add_object()` 로 장애물을 Scene 에 추가
2. `plan_viz_execute()` 로 장애물 너머 목표 계획+실행 — RViz EE 경로(주황)에 우회 모양이 보인다
3. `ready` 복귀
4. `remove_object()` 로 장애물 제거 — 다음 시나리오와 깔끔히 분리

### 4-1. ready 자세로 초기화

In [ ]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
plan_viz_execute_joint(ready_target, vel=0.3, label='Ready')
time.sleep(1.0)

### 4-2. 시나리오 1 — 벽 넘기

base 로부터 45 cm 앞에 폭 40 cm × **높이 40 cm** 의 벽. 벽이 EE 동선 한가운데에
오도록 시작점 / 목표를 명시한다:

- 시작점: `(0.30, 0.0, 0.30)` — 벽 앞쪽 (base 쪽)
- 목표:   `(0.60, 0.0, 0.30)` — 벽 너머

둘 다 그리퍼 아래(`roll=π`), 같은 z=0.30 평면. 벽 윗선 z=0.40 보다 EE 가 낮은 위치에
있으므로 좌→우 직선으로는 갈 수 없고, 플래너가 **벽 위로 호를 그리며** 회피한다.

ready 에서 곧장 목표로 가면 (이전 버전처럼) EE 가 처음부터 z>0.5 영역에서
움직여 벽과 무관한 동선이 되므로, 시작점을 벽 가까이 명시해 학습 효과를 살린다.

In [ ]:
WALL_X, WALL_Z, WALL_H = 0.45, 0.20, 0.40
wall = make_box('wall', position=(WALL_X, 0.0, WALL_Z),
                dimensions=(0.04, 0.40, WALL_H))
add_object(wall)
node.get_logger().info(f'  벽 추가 (x={WALL_X}, 높이 {WALL_H*100:.0f} cm, 폭 40 cm)')

start_pos1 = (0.30, 0.0, 0.30)   # 벽 앞쪽
goal_pos1  = (0.60, 0.0, 0.30)   # 벽 너머
publish_endpoints(start_pos1, goal_pos1, 'Wall Goal', COLOR_GOAL_RED)
time.sleep(1.0)

start1 = make_pose(*start_pos1, math.pi, 0.0, 0.0)
goal1  = make_pose(*goal_pos1,  math.pi, 0.0, 0.0)

node.get_logger().info('--- 벽 앞쪽 시작점으로 이동 ---')
plan_viz_execute(start1, label='WallStart')
time.sleep(1.0)

node.get_logger().info('--- 벽 너머 목표로 이동 (벽 위로 회피) ---')
ok = plan_viz_execute(goal1, label='OverWall')
node.get_logger().info(f'  결과: {"성공 (벽 위 회피)" if ok else "실패"}')
time.sleep(1.0)

plan_viz_execute_joint(ready_target, label='Ready')
time.sleep(0.5)
remove_object('wall')
node.get_logger().info('  벽 제거됨')
time.sleep(0.5)

### 4-3. 시나리오 2 — 사각 기둥 우회 (y 좌우 가로지름)

좌측 (`y=+0.15`) 에 폭 8 cm × 높이 60 cm 기둥. 시작점은 정면 (`y=0.0`),
목표는 기둥 너머 좌측 (`y=+0.30`). 직선 이동이면 (`y=0.0 → +0.30`) 기둥을 *통과* 해야
하므로 플래너는 앞쪽이나 위쪽으로 우회 경로를 만든다.

In [ ]:
pillar = make_box('pillar', position=(0.45, 0.15, 0.30),
                  dimensions=(0.08, 0.08, 0.60))
add_object(pillar)
node.get_logger().info('  기둥 추가 (x=0.45, y=+0.15, 폭 8 cm, 높이 60 cm)')

start_pos2 = (0.45,  0.00, 0.30)
goal_pos2  = (0.45,  0.30, 0.30)
publish_endpoints(start_pos2, goal_pos2, 'Pillar Goal', COLOR_GOAL_BLUE)
time.sleep(1.0)

start2 = make_pose(*start_pos2, math.pi, 0.0, 0.0)
goal2  = make_pose(*goal_pos2,  math.pi, 0.0, 0.0)

node.get_logger().info('--- 기둥 앞쪽 시작점으로 이동 ---')
plan_viz_execute(start2, label='PillarStart')
time.sleep(1.0)

node.get_logger().info('--- 기둥 너머 좌측 목표로 이동 (우회) ---')
ok = plan_viz_execute(goal2, label='AroundPillar')
node.get_logger().info(f'  결과: {"성공 (우회 경로)" if ok else "실패"}')
time.sleep(1.0)

plan_viz_execute_joint(ready_target, label='Ready')
time.sleep(0.5)
remove_object('pillar')
node.get_logger().info('  기둥 제거됨')
time.sleep(0.5)

### 4-4. 시나리오 3 — 정면 원기둥 회피 (y 좌우 가로지름)

정면 중앙 (`x=0.50, y=0.0`) 에 반경 6 cm × 높이 50 cm 원기둥. 시작점은 우측 (`y=-0.20`),
목표는 좌측 (`y=+0.20`). 끝단이 좌→우 직선으로 가려면 원기둥을 통과해야 하므로
플래너가 한쪽으로 둥글게 우회한다.

In [ ]:
cylinder = make_cylinder('cyl', position=(0.50, 0.0, 0.30),
                          height=0.50, radius=0.06)
add_object(cylinder)
node.get_logger().info('  원기둥 추가 (x=0.50, y=0.0, r=6 cm, h=50 cm)')

start_pos3 = (0.50, -0.20, 0.30)
goal_pos3  = (0.50,  0.20, 0.30)
publish_endpoints(start_pos3, goal_pos3, 'Cyl Goal', COLOR_GOAL_PURPLE)
time.sleep(1.0)

start3 = make_pose(*start_pos3, math.pi, 0.0, 0.0)
goal3  = make_pose(*goal_pos3,  math.pi, 0.0, 0.0)

node.get_logger().info('--- 원기둥 우측 시작점으로 이동 ---')
plan_viz_execute(start3, label='CylStart')
time.sleep(1.0)

node.get_logger().info('--- 원기둥 좌측 목표로 이동 (우회) ---')
ok = plan_viz_execute(goal3, label='AroundCylinder')
node.get_logger().info(f'  결과: {"성공 (회피)" if ok else "실패"}')
time.sleep(1.0)

### 4-5. `clear_all()` + `ready` 복귀

In [ ]:
plan_viz_execute_joint(ready_target, label='Ready')
clear_all()
node.get_logger().info('=== franka_ex09 완료 — 모든 객체 제거됨 ===')

## 5. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass